# MAF Replication on the MemeDecode Dataset — GPU run

Runs the **same code** as the local `Replication/Scripts/` folder, on a Colab GPU.

Training MAF fine-tunes all ~110M parameters of Bangla-BERT (CLIP stays frozen). On CPU that is a multi-day run; on a free Colab T4 the paper's configuration (batch 4, 20 epochs) finishes in roughly 1–2 hours.

**Before you start:** set the runtime to a GPU — *Runtime → Change runtime type → T4 GPU*.

### What to upload

OCR (Stage 1) has already been run locally, so its output ships with the dataset and does **not** need to be repeated here. Upload a **zip file**, not the folder — Drive uploads 3,303 individual images very slowly, and Colab reading them back through the Drive mount is slow again. Build it on your machine with:

```powershell
powershell -File Scripts\make_colab_zip.ps1
```

(Do **not** use `Compress-Archive` — it writes backslash path separators that Linux `unzip` cannot read as folders.) Then put the resulting `Replication.zip` in your Google Drive at:

```
MyDrive/MemeDecode/Replication.zip
```

It must contain `Dataset/Img/`, `Dataset/*.csv`, `Scripts/*.py` and `requirements.txt`.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'

## 2. Mount Drive and unpack the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/MemeDecode/Replication.zip'

import os
assert os.path.isfile(ZIP), f'Not found: {ZIP}  — upload the zip to Drive first.'
!rm -rf /content/Replication
!unzip -q "$ZIP" -d /content/
# Handle both zip layouts: one that contains Replication/ and one zipped from inside it.
if not os.path.isdir('/content/Replication') and os.path.isdir('/content/Dataset'):
    os.makedirs('/content/Replication', exist_ok=True)
    !mv /content/Dataset /content/Scripts /content/requirements.txt /content/Replication/ 2>/dev/null

ROOT = '/content/Replication'
print(sorted(os.listdir(ROOT)))
print('memes  :', len(os.listdir(f'{ROOT}/Dataset/Img')))

## 3. Install dependencies

Colab already ships a CUDA build of torch, so only the extras are installed here — notably CLIP from source, exactly as the original `requirements.txt` specifies.

In [ ]:
!pip install -q transformers sentencepiece imbalanced-learn madgrad ftfy regex
!pip install -q git+https://github.com/openai/CLIP.git

import clip, transformers
print('transformers', transformers.__version__)
print('clip OK')

## 4. Rebuild the splits (optional)

The three split CSVs are already in the zip. Rerun this only if you want to change the seed or re-derive them — it needs `Train/` and `Test/` from the original dataset, so leave it skipped otherwise.

This cell also prints the paper's Tables 1–3 for the splits you are actually training on.

In [ ]:
import pandas as pd
for name in ['training_set', 'validation_set', 'testing_set']:
    df = pd.read_csv(f'{ROOT}/Dataset/{name}.csv')
    print(f'{name:<16} {len(df):>5} rows   ', dict(df["Label"].value_counts()))

## 5. Smoke test

Runs the full pipeline on a handful of memes for one epoch. This is **not a result** — it only proves the data, the model and the metrics all wire up before committing to a long run.

In [ ]:
%cd {ROOT}/Scripts
!python main.py --subset 24 --n_iter 1 --run_name colab_smoketest

## 6. The real run

Paper hyperparameters (Appendix A): batch 4, 20 epochs, lr 5e-5, 16 attention heads, max_len 70.

The best-validation-accuracy checkpoint is kept and used for the test evaluation, as in the original.

In [ ]:
%cd {ROOT}/Scripts
!python main.py --run_name maf_full --batch_size 4 --n_iter 20 --lrate 5e-5 --heads 16 --max_len 70

## 7. Ablations and variants (optional)

- `--attn_variant paper` — the attention operand order the **paper text** describes (Q from text, K/V from vision), rather than the order the **released code** implements. See README §5.4.
- `--fix_scheduler` — steps the LR scheduler per batch instead of per epoch, correcting the original's scheduler bug. See README §5.5.

In [ ]:
%cd {ROOT}/Scripts
!python main.py --run_name maf_paper_attn --attn_variant paper
!python main.py --run_name maf_fixed_sched --fix_scheduler

## 8. Results

Compares every run in `Outputs/` against the paper's published MAF row.

**These numbers are not directly comparable to the paper's** — our task is 4-way rather than 5-way, and our captions are raw OCR with no manual correction pass. See README §5.

In [ ]:
import glob, json
import pandas as pd

rows = []
for path in sorted(glob.glob(f'{ROOT}/Outputs/results_*.json')):
    r = json.load(open(path, encoding='utf-8'))
    rows.append({
        'run': r['run_name'],
        'Acc': round(r['accuracy'], 3),
        'WF1': round(r['weighted_f1'], 3),
        'MacroF1': round(r['macro_f1'], 3),
        'MMAE': round(r['mmae'], 3),
    })
rows.append({'run': 'PAPER MAF (5-way MIMOSA)', 'Acc': 0.741, 'WF1': 0.742, 'MacroF1': None, 'MMAE': 0.645})
print(pd.DataFrame(rows).to_string(index=False))

## 9. Confusion matrix

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns

RUN = 'maf_full'
r = json.load(open(f'{ROOT}/Outputs/results_{RUN}.json', encoding='utf-8'))

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(r['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=r['target_names'], yticklabels=r['target_names'], cbar=False)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'MAF — {RUN}')
plt.tight_layout(); plt.show()

## 10. Save results back to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/MemeDecode/results
!cp -r {ROOT}/Outputs/* /content/drive/MyDrive/MemeDecode/results/ 2>/dev/null
!cp {ROOT}/Saved_Models/maf_model_*.pth /content/drive/MyDrive/MemeDecode/results/ 2>/dev/null
!ls -lh /content/drive/MyDrive/MemeDecode/results/

## 11. Generate the Kaggle submission

Produces `submission.csv` from the trained checkpoint, in the exact format
`sample_submission.csv` requires (`Image_name,Target`, using the original
`Neutral/Genders/Politics/Religion` vocabulary). This script never touches ground-truth
labels - it only reads images + captions and writes out what the model predicts.

Upload the `sample_submission.csv` that came with the competition to Drive first, or
point `--sample_submission` at wherever you put it.

Each run now saves its own checkpoint, `Saved_Models/maf_model_<run_name>.pth` — earlier
versions of this notebook wrote every run to one shared `maf_model.pth`, so later runs
overwrote earlier models. The cell below submits `maf_full`; change the name to submit
another run.

The cell after it needs **no checkpoint at all**: every `predictions_<run>.csv` already
holds that run's prediction for all 400 submission images, so
`predictions_to_submission.py` turns each one into `submission_<run>.csv`.

In [ ]:
%cd {ROOT}/Scripts
!python generate_submission.py --checkpoint {ROOT}/Saved_Models/maf_model_maf_full.pth --sample_submission /content/drive/MyDrive/MemeDecode/sample_submission.csv --out {ROOT}/Outputs/submission.csv

import pandas as pd
pd.read_csv(f'{ROOT}/Outputs/submission.csv').head()

In [ ]:
%cd {ROOT}/Scripts
# One submission_<run>.csv per finished run, straight from its predictions CSV (no checkpoint needed).
!python predictions_to_submission.py --predictions "{ROOT}/Outputs/predictions_maf_*.csv" --sample_submission /content/drive/MyDrive/MemeDecode/sample_submission.csv

In [ ]:
!mkdir -p /content/drive/MyDrive/MemeDecode/results
!cp {ROOT}/Outputs/submission*.csv /content/drive/MyDrive/MemeDecode/results/
print("Saved to Drive. Download a submission file from there and upload it on the "
      "Kaggle competition's Submit Predictions page.")